installation part 

In [ ]:
%pip install langchain_community langchain_google_genai langchain_text_splitters langchain_core pydantic faiss-cpu pypdf python-dotenv

import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI , GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate

step 1 :- chuncking 

In [ ]:
docs = (
    PyPDFLoader("./documents/Company_Policies.pdf").load()
    + PyPDFLoader("./documents/Company_Profile.pdf").load()
    + PyPDFLoader("./documents/Product_and_Pricing.pdf").load()
)

In [ ]:
chunks = RecursiveCharacterTextSplitter(
    chunk_size=600, chunk_overlap=150
).split_documents(docs)
print(f"Total Chunks: {len(chunks)}")

In [ ]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vector_store = FAISS.from_documents(chunks, embeddings)

In [67]:
vector_store.index_to_docstore_id

{0: 'a43832e8-981c-4756-8d4d-fd7f77dbd602',
 1: '367ffd78-2d42-4e29-9aeb-e81724ab309a',
 2: '4df65469-456c-49df-af3c-2173dce3dd70',
 3: 'a54eaed8-5aea-48b2-9aff-8f461c2c386f',
 4: '036562b2-560b-4cdd-a02c-093fda0e1326',
 5: '52b87e26-404b-478e-b9ad-e9794a3cd51d',
 6: '1319a5f0-b036-4008-aabe-153f3ef2a6e0',
 7: '1bfedece-b48d-4313-9d6c-4f15ad6d429e',
 8: 'b4cf85f8-e8c7-49f1-b262-85e7ec03a021',
 9: 'b855f9f0-e6b6-4392-bc6b-702e87470af0',
 10: 'dbc4dbcd-1eb2-4ba1-a66f-46005a09948c',
 11: '11d6d904-7c31-4cac-ac17-9526f6ba9ae0',
 12: 'a96d8762-5c21-45a5-8d2f-abf310d7986b',
 13: 'a612731b-76c8-493c-8f7d-5b4bccf7c237',
 14: '070468bd-0aac-4f17-9547-514a99db286a',
 15: '734a62ef-4f86-482d-bcbc-13122b738eef'}

step2 :- Reterival

In [70]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12b3f2450>, search_kwargs={'k': 4})

In [71]:
retriever.invoke('who is the CEO of the company?')

[Document(id='1319a5f0-b036-4008-aabe-153f3ef2a6e0', metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-14T13:07:17+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-14T13:07:17+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './documents/Company_Profile.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='Founder\nAarav Mehta founded NexaAI after over 10 years of experience in enterprise data platforms and\ncloud infrastructure.\nHe previously worked with global consulting firms where he led multiple large-scale digital\ntransformation projects.\nLeadership Team\nThe leadership team brings experience across AI engineering, product management, and business\noperations.\n\x7f\nAarav Mehta – CEO & Founder\n\x7f\nRiya Kapoor – CTO (Distributed systems & AI platforms)\n\x7f\nKunal Sharma – Head of Product\n\x7f\nNeha Verma – Head of Operations &

Step 3 - Augmentation

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question = "tell me about the company?"
retrieved_docs    = retriever.invoke(question)

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)